# Regression model comparison

Run the five individual model notebooks first. This notebook loads their fitted pipelines and repeats the same training-fold comparison for their selected configurations. The results stay here; it exports only the two selected fitted pipelines and their small evaluation metadata files under `models/`.

MAE remains the original selection rule. Both scenarios use the same students and validation folds. Selection reused these folds, so scores may be optimistic. The held-out results have already been inspected in Notebook 04 and must not guide new tuning.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


## 1 Compare the selected configuration of each model

Each cloned pipeline is fitted on four folds and evaluated on the fifth. The loaded fitted pipeline is not changed. Fold-average grade metrics are shown alongside pooled risk metrics; the latter sum the confusion counts across all 316 training students. Accuracy alone can hide missed students.

In [2]:
MODELS = {
    "linear_regression": "Linear Regression", "ridge": "Ridge",
    "decision_tree": "Decision Tree", "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
}
SCENARIOS = {"no_prior_grades": X_no_grades, "with_prior_grades": X_with_grades}
comparison_rows, evaluated_folds = [], {}
for scenario, X_train in SCENARIOS.items():
    for slug, name in MODELS.items():
        pipeline = joblib.load(PROJECT_ROOT / "models/experiments" / f"{slug}__{scenario}.joblib")
        scores = cross_validate(pipeline, X_train, y_train, cv=CV, scoring=SCORING,
            n_jobs=-1, error_score="raise")
        folds = fold_results(scores)
        evaluated_folds[(scenario, slug)] = folds
        row = {"Scenario": scenario, "Model": name, "slug": slug}
        row.update(folds[["MAE", "RMSE", "R2"]].mean().to_dict())
        row.update(pooled_risk(folds).to_dict())
        comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows)
for scenario in SCENARIOS:
    print("Scenario:", scenario)
    display(comparison.loc[comparison.Scenario.eq(scenario)].sort_values("MAE").drop(columns="slug").round(4))

,Scenario,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
3,no_prior_grades,Random Forest,2.9641,3.9366,0.2081,0.6930,0.5319,0.4854,0.5076,0.7934,50.0,44.0,53.0,169.0
4,no_prior_grades,Gradient Boosting,3.0461,3.9459,0.2102,0.6899,0.5269,0.4757,0.5000,0.7934,49.0,44.0,54.0,169.0
2,no_prior_grades,Decision Tree,3.0791,4.0636,0.1549,0.6804,0.5161,0.3107,0.3879,0.8592,32.0,30.0,71.0,183.0
1,no_prior_grades,Ridge,3.2423,4.1677,0.1253,0.6551,0.4737,0.5243,0.4977,0.7183,54.0,60.0,49.0,153.0
0,no_prior_grades,Linear Regression,3.4763,4.4615,-0.0308,0.6614,0.4848,0.6214,0.5447,0.6808,64.0,68.0,39.0,145.0


,Scenario,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
8,with_prior_grades,Random Forest,0.9381,1.4757,0.8928,0.8703,0.7183,0.9903,0.8327,0.8122,102.0,40.0,1.0,173.0
9,with_prior_grades,Gradient Boosting,0.9849,1.5800,0.8765,0.8703,0.7183,0.9903,0.8327,0.8122,102.0,40.0,1.0,173.0
7,with_prior_grades,Decision Tree,1.0108,1.6618,0.8646,0.8892,0.7742,0.9320,0.8458,0.8685,96.0,28.0,7.0,185.0
6,with_prior_grades,Ridge,1.2955,1.8116,0.8358,0.8513,0.7000,0.9515,0.8066,0.8028,98.0,42.0,5.0,171.0
5,with_prior_grades,Linear Regression,1.3724,1.8829,0.8239,0.8608,0.7122,0.9612,0.8182,0.8122,99.0,40.0,4.0,173.0


Scenario: no_prior_grades
Scenario: with_prior_grades


## 2 Preserve the selected models for final evaluation

Select the lowest training-CV MAE separately in each scenario. The models are already fitted on all training students, so copy those pipelines for Notebook 04. Small metadata files retain the exact split, feature columns and fingerprints needed to avoid loading the wrong data or model. These are model handoff files, not exported experiment reports.

In [3]:
import json
import hashlib
import platform
import importlib.metadata
import shutil
import re

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for scenario, X_train in SCENARIOS.items():
    winner = comparison.loc[comparison.Scenario.eq(scenario)].sort_values("MAE").iloc[0]
    source_path = PROJECT_ROOT / "models/experiments" / f"{winner.slug}__{scenario}.joblib"
    fitted = joblib.load(source_path)
    model_type = re.sub(r"(?<!^)(?=[A-Z])", "_", type(fitted.named_steps["model"]).__name__).lower()
    grade_scope = "without_prior_grades" if scenario == "no_prior_grades" else "with_prior_grades"
    stem = f"best_{grade_scope}_{model_type}"
    target_path = PROJECT_ROOT / "models" / f"{stem}.joblib"
    shutil.copy2(source_path, target_path)
    fitted = joblib.load(target_path)
    selected_folds = evaluated_folds[(scenario, winner.slug)]
    model_metadata = {
        "model": winner.Model, "scenario": scenario,
        "students": len(df), "zero_grades": int(df.G3.eq(0).sum()),
        "training_indices": train_ids.tolist(), "test_indices": test_ids.tolist(),
        "input_features": X_train.columns.tolist(),
        "include_prior_grades": scenario == "with_prior_grades",
        "selection_metric": "MAE",
        "selection_cv_scores": selected_folds[["MAE", "RMSE", "R2"]].mean().to_dict(),
        "selection_cv_std": selected_folds[["MAE", "RMSE", "R2"]].std().to_dict(),
        "pipeline_path": target_path.relative_to(PROJECT_ROOT).as_posix(),
        "pipeline_sha256": sha256(target_path),
        "dataset_sha256": sha256(PROJECT_ROOT / "data/student-mat.csv"),
        "preprocessing_sha256": sha256(PROJECT_ROOT / "src/preprocessing.py"),
        "log_absences": fitted.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"],
        "versions": {"python": platform.python_version(), **{package: importlib.metadata.version(package)
            for package in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]}},
        "split_random_state": 42, "test_size": 0.2,
        "test_set_already_inspected": True,
    }
    (PROJECT_ROOT / "models" / f"{stem}_metadata.json").write_text(
        json.dumps(model_metadata, indent=2), encoding="utf-8")
    print(scenario, "→", winner.Model, "| Training CV MAE:", round(winner.MAE, 4))

no_prior_grades → Random Forest | Training CV MAE: 2.9641
with_prior_grades → Random Forest | Training CV MAE: 0.9381


## 3 Interpret the comparison

Small differences between candidates are not proof of a statistically significant improvement. A good grade predictor may still miss many at-risk students at the fixed threshold. Inspect recall, precision and false negatives alongside MAE.

The G1/G2 scenario predicts at a later time. Availability of absences and other variables remains unresolved. Full-dataset EDA preceded splitting. Use Notebook 04 for the two frozen selected models' final test results, residuals and recommendations; do not use those test results to revise this ranking.